In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

warehouse_dir = Path("/home/jovyan/work/spark-warehouse/iceberg").resolve()
warehouse_dir.mkdir(parents=True, exist_ok=True)
caminho_base = f"file://{warehouse_dir.as_posix()}"

# ==========================================
# 1. LIGANDO O MOTOR DO ICEBERG
# ==========================================
iceberg_package = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3"
print("Baixando pacotes e iniciando o motor ICEBERG (Pode demorar uns segundos)...")
builder = SparkSession.builder.appName("Iceberg_Lab") \
    .config("spark.jars.packages", iceberg_package) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.meucatalogo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.meucatalogo.type", "hadoop") \
    .config("spark.sql.catalog.meucatalogo.warehouse", caminho_base)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print(f"Iceberg runtime package: {iceberg_package}")
print("✅ Motor ICEBERG ligado!\n")

# ==========================================
# 2. LABORATÓRIO: DDL, INSERT, UPDATE E DELETE (Via SQL)
# ==========================================
print("--- Criando Banco e Tabelas (DDL) ---")
spark.sql("CREATE NAMESPACE IF NOT EXISTS meucatalogo.faculdade")

spark.sql("""
CREATE TABLE IF NOT EXISTS meucatalogo.faculdade.notas (
    id_nota INT,
    id_aluno INT,
    disciplina STRING,
    valor_nota FLOAT
) USING iceberg
""")

print("--- 1. INSERT ---")
spark.sql("""
INSERT INTO meucatalogo.faculdade.notas VALUES
(101, 1, 'Arquitetura de Dados', 8.5),
(102, 2, 'Arquitetura de Dados', 6.0)
""")
spark.sql("SELECT * FROM meucatalogo.faculdade.notas").show()

print("--- 2. UPDATE (Mudando nota do aluno 1 para 9.5) ---")
spark.sql("""
UPDATE meucatalogo.faculdade.notas
SET valor_nota = 9.5
WHERE id_aluno = 1
""")
spark.sql("SELECT * FROM meucatalogo.faculdade.notas").show()

print("--- 3. DELETE (Removendo o aluno 2) ---")
spark.sql("""
DELETE FROM meucatalogo.faculdade.notas
WHERE id_aluno = 2
""")
spark.sql("SELECT * FROM meucatalogo.faculdade.notas").show()

Baixando pacotes e iniciando o motor ICEBERG (Pode demorar uns segundos)...
Spark version: 3.5.0
Iceberg runtime package: org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3
✅ Motor ICEBERG ligado!

--- Criando Banco e Tabelas (DDL) ---
--- 1. INSERT ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    101|       1|Arquitetura de Dados|       8.5|
|    102|       2|Arquitetura de Dados|       6.0|
+-------+--------+--------------------+----------+

--- 2. UPDATE (Mudando nota do aluno 1 para 9.5) ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    101|       1|Arquitetura de Dados|       9.5|
|    102|       2|Arquitetura de Dados|       6.0|
+-------+--------+--------------------+----------+

--- 3. DELETE (Removendo o aluno 2) ---
+-------+--------+----------------